# Merge daily floorsheet CSVs into a single Parquet file

Reads every `data/raw/floorsheet_*.csv` file, concatenates them, and writes the result to `data/processed/floorsheet.parquet`.

Set input/output paths.

In [1]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")
OUT_PATH = Path("../data/processed/floorsheet.parquet")

List all raw CSV files.

In [2]:
csv_files = sorted(RAW_DIR.glob("floorsheet_*.csv"))
print(f"Found {len(csv_files)} files")
csv_files[:5]

Found 561 files


[PosixPath('../data/raw/floorsheet_2024-01-01.csv'),
 PosixPath('../data/raw/floorsheet_2024-01-02.csv'),
 PosixPath('../data/raw/floorsheet_2024-01-03.csv'),
 PosixPath('../data/raw/floorsheet_2024-01-04.csv'),
 PosixPath('../data/raw/floorsheet_2024-01-07.csv')]

Merge all CSVs into one DataFrame.

In [3]:
dtypes = {
    "transaction": "string",
    "symbol": "string",
    "buyer": "string",
    "seller": "string",
    "quantity": "float64",
    "rate": "float64",
    "amount": "float64",
}

# read and stack all 561 daily CSVs into one DataFrame
df = pd.concat(
    (pd.read_csv(f, dtype=dtypes, parse_dates=["date"]) for f in csv_files),
    ignore_index=True,
)
df.shape

(47346003, 8)

Preview the data.

In [4]:
df.head()

,transaction,symbol,buyer,seller,quantity,rate,amount,date
0,2024010103010653,GBBL,22,53,50.0,410.0,20500.0,2024-01-01
1,2024010105005330,NIBLPF,26,10,5647.0,9.3,52517.1,2024-01-01
2,2024010104015466,PRIN,20,34,10.0,851.0,8510.0,2024-01-01
3,2024010101071456,GVL,3,34,200.0,418.5,83700.0,2024-01-01
4,2024010103010652,HLI,39,34,10.0,438.0,4380.0,2024-01-01


Check dtypes and memory usage.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 47346003 entries, 0 to 47346002
Data columns (total 8 columns):
 #   Column       Dtype         
---  ------       -----         
 0   transaction  string        
 1   symbol       string        
 2   buyer        string        
 3   seller       string        
 4   quantity     float64       
 5   rate         float64       
 6   amount       float64       
 7   date         datetime64[us]
dtypes: datetime64[us](1), float64(3), string(4)
memory usage: 2.8 GB


Save merged data as parquet.

In [7]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT_PATH, index=False)
print(f"Wrote {len(df):,} rows to {OUT_PATH}")

Wrote 47,346,003 rows to ../data/processed/floorsheet.parquet
